In [ ]:
%%javascript
IPython.OutputArea.auto_scroll_threshold = 9999;

In [1]:
from IPython.core.interactiveshell import InteractiveShell  
InteractiveShell.ast_node_interactivity = "all"
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>")) 

In [2]:
from dash import Dash, html, dcc, Input, Output, dash_table, callback, State, ClientsideFunction
from dash.exceptions import PreventUpdate
import dash_bootstrap_components as dbc
import collections
import pandas as pd
import wget
import json
import random
from application.dash.sidebar import spes_options, uga_options, ciblage_options 
from dash_extensions.javascript import assign, arrow_function, Namespace
import dash_leaflet as dl
import dash_leaflet.express as dlx
import re
from datetime import datetime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship, Session
from flask_sqlalchemy import SQLAlchemy
from sqlalchemy import Column, Integer, Sequence, String, Text, DateTime, ForeignKey, create_engine
import datetime
from flask import Flask, request, url_for, render_template, render_template_string
from datatables import DataTable
import io

In [3]:
from application.dash.biocodex.functions import build_modal, build_tile_front, doctor_colors, p_popup, c_popup, data_to_geojson, df
from application.dash.biocodex.functions import prepare_data, join_id_adr_cdb
from application.dash.biocodex.functions import mean_lat, mean_lon, datatable_cols, styles, legend1, legend2

In [4]:
from app import app
app.app_context().push()

In [ ]:
from application.config import basedir
from dotenv import load_dotenv
import os

load_dotenv(os.path.join(basedir, '.env'))
DATABASE_URL = os.environ.get('DATABASE_URL').replace('postgres://', 'postgresql://')
engine = create_engine(DATABASE_URL)

In [5]:
ugas = ["75AUT", "75PAS", "75TRO", "75INV", "75ELY", "75GRE", "75VAU", "75MNP", "75PER", "75TER", "92LEV", "92NEU"]
spes = ["GY", "MGY", "SF", "MG", "GE", "PE", "PPSY", "PSY", "NE", "PHA"]

In [11]:
ns = Namespace('dashExtensions','default')

with open("assets/uga_gpd.json", "r") as uga_json:
    uga_geojson = json.loads(uga_json.read())

sector_features = [
    uga_geojson["features"][i] for i in range(len(uga_geojson["features"])) if uga_geojson["features"][i]["properties"]["CODE_UGA"] in ugas
]
uga_geojson["features"] = sector_features

ugas_layer = dl.GeoJSON(
    data=uga_geojson,
    hideout=dict(selected=ugas),
    filter=ns('uga_geojson_filter'),
    style=ns('uga_style_handle'),
    hoverStyle=arrow_function(dict(weight=3, color='red', dashArray='')),
    zoomToBounds=True,
    id="uga-geojson",
)

dff = prepare_data(df)
data = dff.to_dict('records')
data_geojson = data_to_geojson(data)

data_layer = dl.GeoJSON(
    data=data_geojson,
    hideout=dict(ugas_selected=ugas, spes_selected=spes, cib_selected=[1,2,3,4], pvm_range=[0,49]),
    filter=ns('geojson_filter'),
    zoomToBounds=True,
    pointToLayer=ns('cible_icon'),
    id="data-geojson"
)

In [6]:
from application.dash.biocodex.functions import get_pharmas, pharmas_to_geojson

In [83]:
pharmas = get_pharmas()
pharma_geojson = pharmas_to_geojson(pharmas.to_dict('records'))

pharma_layer = dl.GeoJSON(
    data=pharma_geojson,
    hideout=dict(ugas_selected=ugas, cib_selected=["1","2","3","4"]),
    filter=ns('pharmas_filter'),
    zoomToBounds=True,
    pointToLayer=ns('pharmas_icon'),
    id="pharma-geojson"
)

In [84]:
pharmas[~pharmas["ddv"].isna()].sample()

,cip,nom,uga,adr,cp,ville,tel,cib_vm,cib_dp,cib_dso,...,ca_ul_cma_juin_23,rang_ca_ul_juin23,ca_circ_cma_sept23,ca_ul_cma_sept23,rang_ca_ul_sept23,decil_23,groupement,contrat_23,lat,lon
195,2052043,PHARMACIE BOSQUET GRENELLE,75INV,49 AVENUE BOSQUET,75007,PARIS,01 45 51 35 91,,1,1,...,748,,167,708,,4.0,ALPHEGA DIAMANT,CA<7500 STAR,48.85711,2.304569


In [92]:
ts = pharmas.loc[195, "ddv"]
ts

Timestamp('2022-05-18 00:00:00')

In [95]:
isinstance(ts, pd.Timestamp)

True

In [93]:
ts.to_pydatetime()

datetime.datetime(2022, 5, 18, 0, 0)

In [94]:
ts.strftime(format="%d/%m/%Y %H:%M")

'18/05/2022 00:00'

In [77]:
ph = pharmas.sample()

In [67]:
kpis_cols = [
    'ca_circ_cma_fev23', 'rang_ca_circ_fev23', 'ca_ul_cma_fev_23', 'rang_ca_ul_fev23',
    'ca_circ_cma_juin23', 'rang_ca_circ_juin23', 'ca_ul_cma_juin_23', 'rang_ca_ul_juin23', 
    'ca_circ_cma_sept23', 'ca_ul_cma_sept23', 'rang_ca_ul_sept23'
]

In [78]:
ph = ph[kpis_cols]

In [79]:
mux = pd.MultiIndex.from_product([['fev23', 'juin23','sept23'], ['cma','rank']])
df = pd.DataFrame(columns=mux)

In [80]:
df.loc['circadin', ('fev23', 'cma')] = ph['ca_circ_cma_fev23'].values[0]
df.loc['circadin', ('juin23', 'cma')] = ph['ca_circ_cma_juin23'].values[0]
df.loc['circadin', ('sept23', 'cma')] = ph['ca_circ_cma_sept23'].values[0]
df.loc['circadin', ('fev23', 'rank')] = ph['rang_ca_circ_fev23'].values[0]
df.loc['circadin', ('juin23', 'rank')] = ph['rang_ca_circ_juin23'].values[0]
df.loc['ul', ('fev23', 'cma')] = ph['ca_ul_cma_fev_23'].values[0]
df.loc['ul', ('juin23', 'cma')] = ph['ca_ul_cma_juin_23'].values[0]
df.loc['ul', ('sept23', 'cma')] = ph['ca_ul_cma_sept23'].values[0]
df.loc['ul', ('fev23', 'rank')] = ph['rang_ca_ul_fev23'].values[0]
df.loc['ul', ('juin23', 'rank')] = ph['rang_ca_ul_juin23'].values[0]
df.loc['ul', ('sept23', 'rank')] = ph['rang_ca_ul_sept23'].values[0]
df

fev23      juin23      sept23     
           cma rank    cma rank    cma rank
circadin   714   81    334         624  NaN
ul        2082   58   2050   59   1858   66

In [82]:
generate_html_table_from_df(df)

[Thead([Tr([Th(('fev23', 'cma')), Th(('fev23', 'rank')), Th(('juin23', 'cma')), Th(('juin23', 'rank')), Th(('sept23', 'cma')), Th(('sept23', 'rank'))])]),
 Tbody([Tr([Td(children='714', id='<built-in function id>_0_0'), Td(children='81', id='<built-in function id>_0_1'), Td(children='334', id='<built-in function id>_0_2'), Td(children='', id='<built-in function id>_0_3'), Td(children='624', id='<built-in function id>_0_4'), Td(children=nan, id='<built-in function id>_0_5')]), Tr([Td(children='2082', id='<built-in function id>_1_0'), Td(children='58', id='<built-in function id>_1_1'), Td(children='2050', id='<built-in function id>_1_2'), Td(children='59', id='<built-in function id>_1_3'), Td(children='1858', id='<built-in function id>_1_4'), Td(children='66', id='<built-in function id>_1_5')])])]

In [54]:
pharmas.columns

Index(['cip', 'nom', 'uga', 'adr', 'cp', 'ville', 'tel', 'cib_vm', 'cib_dp',
       'cib_dso', 'nv22', 'ddv', 'rdv', 'ca_circ_cma_fev23',
       'rang_ca_circ_fev23', 'ca_ul_cma_fev_23', 'rang_ca_ul_fev23',
       'ca_circ_cma_juin23', 'rang_ca_circ_juin23', 'ca_ul_cma_juin_23',
       'rang_ca_ul_juin23', 'ca_circ_cma_sept23', 'ca_ul_cma_sept23',
       'rang_ca_ul_sept23', 'decil_23', 'groupement', 'contrat_23', 'lat',
       'lon'],
      dtype='object')

In [51]:
generate_html_table_from_df(pharmas.sample())

[Thead([Tr([Th('cip'), Th('nom'), Th('uga'), Th('adr'), Th('cp'), Th('ville'), Th('tel'), Th('cib_vm'), Th('cib_dp'), Th('cib_dso'), Th('nv22'), Th('ddv'), Th('rdv'), Th('ca_circ_cma_fev23'), Th('rang_ca_circ_fev23'), Th('ca_ul_cma_fev_23'), Th('rang_ca_ul_fev23'), Th('ca_circ_cma_juin23'), Th('rang_ca_circ_juin23'), Th('ca_ul_cma_juin_23'), Th('rang_ca_ul_juin23'), Th('ca_circ_cma_sept23'), Th('ca_ul_cma_sept23'), Th('rang_ca_ul_sept23'), Th('decil_23'), Th('groupement'), Th('contrat_23'), Th('lat'), Th('lon')])]),
 Tbody([Tr([Td(children=1203971, id='<built-in function id>_0_0'), Td(children='SERVICE MEDICAL DE L ENSAAMA', id='<built-in function id>_0_1'), Td(children='75VAU', id='<built-in function id>_0_2'), Td(children='63 65 RUE OLIVIER DE SERRES', id='<built-in function id>_0_3'), Td(children=75015, id='<built-in function id>_0_4'), Td(children='PARIS', id='<built-in function id>_0_5'), Td(children='01 53 68 16 90', id='<built-in function id>_0_6'), Td(children='', id='<built-in

In [18]:
layout = html.Div(
        [
            dl.Map(
                [
                   dl.LayersControl(
                        [
                            dl.BaseLayer(dl.TileLayer(), name="base", checked=True),
                            dl.Overlay(dl.LayerGroup(ugas_layer, id="ugas_layer_group"), name="ugas", checked=True),
                            dl.Overlay(dl.LayerGroup(data_layer, id="data_layer_group"), name="data", checked=True),
                            dl.Overlay(dl.LayerGroup(pharma_layer, id="pharma_layer_group"), name="pharmas", checked=True),
                            #dl.Overlay(dl.LayerGroup(untarget_layer, id="untarget_layer_group"), name="non ciblés", checked=False),
                            #info
                        ]
                   ),
                    dl.FullScreenControl(),
                    dl.LocateControl(locateOptions={'enableHighAccuracy': True})
               ],
               center=(mean_lat, mean_lon),
               zoom=11,
               style={'height': '90vh', 'position': 'absolute', 'width': '90%'}
            )
        ],
        id='map'
    )

In [19]:
external_scripts=[
    "https://code.jquery.com/jquery-3.7.0.js"
]


external_stylesheets=[
    dbc.themes.BOOTSTRAP,
    dbc.icons.FONT_AWESOME,
    # Leaflet
    {"rel": "stylesheet", "type": "text/css", "src": "css/leaflet.extra-markers.min.css"},
    # Sidebar
    {"rel": "stylesheet","type": "text/css","src": "css/sidebar.css"},
    # Hamburgers
    {"rel": "stylesheet","type": "text/css","src": "css/hamburgers.css"}
]

In [20]:
app = Dash(__name__, external_stylesheets=external_stylesheets, external_scripts=external_scripts)
app.layout=layout
app.run_server(debug=True)

In [21]:
df = join_id_adr_cdb()

In [23]:
sample = df.sample()
sample.columns

Index(['id', 'nom', 'prenom', 'spe', 'pot', 'pvm', 'nv22', 'cib', 'dec', 'eta',
       'uga', 'adr', 'cp', 'ville', 'tel', 'lat', 'lon', 'mode', 'com', 'ddv',
       'dpv', 'rdv', 'rec', 'pk', 'lun_mat', 'lun_am', 'mar_mat', 'mar_am',
       'mer_mat', 'mer_am', 'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am'],
      dtype='object')

In [24]:
sample.drop(['id', 'nom', 'prenom', 'spe', 'pot', 'pvm', 'nv22', 'cib', 'dec', 'eta',
       'uga', 'adr', 'cp', 'ville', 'tel', 'lat', 'lon', 'mode', 'com', 'ddv',
       'dpv', 'rdv', 'rec', 'pk'], axis=1, inplace=True)

In [27]:
sample = sample.T

In [33]:
sample.columns

Index([1650, 'MATIN', 'AP. MIDI'], dtype='object')

In [34]:
sample['MATIN'] = pd.Series(dtype=str)
sample['AP. MIDI'] = pd.Series(dtype=str)
sample.drop(1650, axis=1, inplace=True)
sample.drop(['mer_am', 'jeu_mat', 'jeu_am', 'ven_mat', 'ven_am'], axis=0, inplace=True)
sample

,MATIN,AP. MIDI
lun_mat,NaN,NaN
lun_am,NaN,NaN
mar_mat,NaN,NaN
mar_am,NaN,NaN
mer_mat,NaN,NaN


In [37]:
sample.index = pd.Index(['LUNDI', 'MARDI', 'MERCREDI', 'JEUDI', 'VENDREDI'], dtype='object')

In [38]:
sample

,MATIN,AP. MIDI
LUNDI,NaN,NaN
MARDI,NaN,NaN
MERCREDI,NaN,NaN
JEUDI,NaN,NaN
VENDREDI,NaN,NaN


In [39]:
def generate_html_table_from_df(df):
    Thead = html.Thead(
        [html.Tr([html.Th(col) for col in df.columns])]
    )
    Tbody = html.Tbody(
        [html.Tr(
            [html.Td( df.iloc[i, j], id = '{}_{}_{}'.format(id, i, j) ) for j in range(len(df.columns))]
         ) for i in range(len(df))]
    )
    return [Thead, Tbody]

In [40]:
generate_html_table_from_df(sample)

[Thead([Tr([Th('MATIN'), Th('AP. MIDI')])]),
 Tbody([Tr([Td(children=nan, id='<built-in function id>_0_0'), Td(children=nan, id='<built-in function id>_0_1')]), Tr([Td(children=nan, id='<built-in function id>_1_0'), Td(children=nan, id='<built-in function id>_1_1')]), Tr([Td(children=nan, id='<built-in function id>_2_0'), Td(children=nan, id='<built-in function id>_2_1')]), Tr([Td(children=nan, id='<built-in function id>_3_0'), Td(children=nan, id='<built-in function id>_3_1')]), Tr([Td(children=nan, id='<built-in function id>_4_0'), Td(children=nan, id='<built-in function id>_4_1')])])]

In [43]:
row = df.sample().to_dict('records')[0]
row

{'id': 195,
 'nom': 'NACCACHE',
 'prenom': 'Jean Pierre',
 'spe': 'GY',
 'pot': 162,
 'pvm': 21,
 'nv22': 1,
 'cib': 4,
 'dec': '',
 'eta': 'COSEM',
 'uga': '75ELY',
 'adr': '6 AVENUE CESAR CAIRE',
 'cp': '75008',
 'ville': 'PARIS',
 'tel': '01 55 56 62 51',
 'lat': 48.87638854980469,
 'lon': 2.31978178024292,
 'mode': 'CHECK',
 'com': 'COSEM + 75PER',
 'ddv': nan,
 'dpv': nan,
 'rdv': '',
 'rec': '',
 'pk': '',
 'lun_mat': '',
 'lun_am': '',
 'mar_mat': '',
 'mar_am': '',
 'mer_mat': '',
 'mer_am': '',
 'jeu_mat': '',
 'jeu_am': '',
 'ven_mat': '',
 'ven_am': ''}

In [49]:
build_calendar(row)

[Thead([Tr([Th('MATIN'), Th('AP. MIDI')])]),
 Tbody([Tr([Td(children=Div(children=[FormFloating(children=[Input(className='pt-3', disabled=True, name='adr', size='sm', value='')], className='border-3 border-info col-12 px-0')], className='d-flex'), id='<built-in function id>_0_0'), Td(children=Div(children=[FormFloating(children=[Input(className='pt-3', disabled=True, name='adr', size='sm', value='')], className='border-3 border-info col-12 px-0')], className='d-flex'), id='<built-in function id>_0_1')]), Tr([Td(children=Div(children=[FormFloating(children=[Input(className='pt-3', disabled=True, name='adr', size='sm', value='')], className='border-3 border-info col-12 px-0')], className='d-flex'), id='<built-in function id>_1_0'), Td(children=Div(children=[FormFloating(children=[Input(className='pt-3', disabled=True, name='adr', size='sm', value='')], className='border-3 border-info col-12 px-0')], className='d-flex'), id='<built-in function id>_1_1')]), Tr([Td(children=Div(children=[F